# Workshop 3 · 01 — Stammdaten heilen

In diesem Abschnitt bereiten wir unvollständige und uneinheitliche Stammdaten sowie Freitextmeldungen so auf, dass daraus **saubere, strukturierte Spalten** werden – deklarativ mit **AI Functions**, ohne eigenes Modelltraining.

**Zwei Schritte:**
1. **Normalisieren** — `depot` und `baureihe` mit `ai.classify` auf einheitliche Werte mappen.
2. **Extrahieren** — fehlende Felder (`komponente`, `ort`, `schweregrad`, `sicherheitsrelevant`, `prioritaet`) mit `ai.generate_response` + **JSON-Schema** aus dem Freitext ziehen.

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/overview

In [ ]:
import synapse.ml.spark.aifunc as aifunc  # registriert den .ai Accessor auf Spark-DataFrames

df = spark.table("asset_meldungen")

# Ausgangslage: uneinheitliche/leere Stammdaten, die Information steckt nur im Freitext
display(df.select("meldung_id", "baureihe", "depot", "komponente", "freitext_meldung"))

## Schritt 1 — Stammdaten normalisieren mit `ai.classify`

`depot` (frei geschrieben) und `baureihe` (Tippfehler, Schreibvarianten, teils leer) werden auf **einheitliche Werte** gemappt — eine Zeile Code statt 50 `CASE WHEN`.

In [ ]:
# This code uses AI. Always review output for mistakes.

depot_labels = ["Hamburg-Eidelstedt", "Frankfurt-Griesheim", "Muenchen-Pasing",
                "Leipzig", "Berlin-Rummelsburg"]
baureihe_labels = ["ICE4", "ICE3", "IC2", "Talent2", "Flirt3", "Doppelstock", "unbekannt"]

norm = (df
        .ai.classify(input_col="depot", output_col="depot_norm", labels=depot_labels)
        .ai.classify(input_col="baureihe", output_col="baureihe_norm", labels=baureihe_labels))

display(norm.select("meldung_id", "depot", "depot_norm", "baureihe", "baureihe_norm"))

## Schritt 2 — fehlende Felder aus dem Freitext ziehen

`ai.generate_response` mit einem **JSON-Schema** liefert pro Zeile **typisierte Felder** (kein Freitext-Raten): Komponente, Ort, Schweregrad (1–5), Sicherheitsrelevanz, Priorität. Das ist „unstrukturiert → strukturiert" in einer Zelle.

In [ ]:
# This code uses AI. Always review output for mistakes.

meldung_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "meldung",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "komponente": {"type": "string"},
                "ort_am_fahrzeug": {"type": "string"},
                "schweregrad": {"type": "integer", "description": "1 kosmetisch bis 5 sicherheitskritisch"},
                "sicherheitsrelevant": {"type": "boolean"},
                "prioritaet": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]},
            },
            "required": ["komponente", "ort_am_fahrzeug", "schweregrad", "sicherheitsrelevant", "prioritaet"],
            "additionalProperties": False,
        },
    },
}

extracted = norm.ai.generate_response(
    prompt=("Extrahiere aus dieser Werkstatt-Meldung strukturierte Felder: {freitext_meldung}. "
            "Schweregrad 1 (kosmetisch) bis 5 (sicherheitskritisch)."),
    is_prompt_template=True,
    response_format=meldung_schema,
    output_col="extract_json",
)

display(extracted.select("meldung_id", "freitext_meldung", "extract_json"))

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

ext_schema = StructType([
    StructField("komponente", StringType()),
    StructField("ort_am_fahrzeug", StringType()),
    StructField("schweregrad", IntegerType()),
    StructField("sicherheitsrelevant", BooleanType()),
    StructField("prioritaet", StringType()),
])

clean = (extracted
         .withColumn("e", from_json(col("extract_json"), ext_schema))
         .select("meldung_id", "wagon_id", "baureihe_norm", "depot_norm",
                 col("e.komponente").alias("komponente"),
                 col("e.ort_am_fahrzeug").alias("ort"),
                 col("e.schweregrad").alias("schweregrad"),
                 col("e.sicherheitsrelevant").alias("sicherheitsrelevant"),
                 col("e.prioritaet").alias("prioritaet"),
                 "freitext_meldung", "kunden_kommentar", "foto_pfad"))

clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("asset_clean")
display(clean.orderBy(col("schweregrad").desc()))

### Takeaway

Aus 24 uneinheitlichen Zeilen wurde eine **saubere, typisierte Tabelle** `asset_clean` — normalisierte Stammdaten plus aus Freitext extrahierte Felder. Klassische BI-Ansätze würden hier aufwändige Regex-/Mapping-Logik erfordern.

> **Kurzform für reine Entitäts-Extraktion:** `df.ai.extract(input_col="freitext_meldung", labels=["komponente", "ort", "schweregrad"])` liefert je Label eine Spalte — ideal, wenn kein striktes Schema nötig ist.